# 05 · fastText: subpalavras, morfologia e palavras fora do vocabulário

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


**Artigo-base (leitura complementar):** Bojanowski, Grave, Joulin & Mikolov (2017), *Enriching Word Vectors with Subword Information* (TACL).

## O problema
O word2vec dá um vetor **a cada forma**: *casa*, *casas*, *casinha*, *casarão* são tratadas como símbolos sem relação. Em línguas morfologicamente ricas (turco, finlandês, tcheco, russo — e o português, com ~50 formas por verbo) muitas formas são raras ou inéditas no corpus; palavras fora do vocabulário (**OOV**) simplesmente não têm vetor.

## A proposta
Representar cada palavra como o conjunto $\mathcal{G}_w$ dos seus **n-gramas de caracteres** ($3 \le n \le 6$), delimitados por `<` e `>`, mais a palavra inteira. A pontuação do Skip-gram com amostragem negativa vira

$$s(w, c) = \sum_{g \in \mathcal{G}_w} \mathbf{z}_g^{\top} \mathbf{v}_c$$

Os n-gramas são mapeados em $K = 2\cdot10^6$ *buckets* por *hashing* (FNV-1a) para limitar a memória. Consequências: parâmetros compartilhados entre formas aparentadas; vetores para OOV (soma dos n-gramas); treino só ~1,5× mais lento.

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: Y:\BLIS\linguagemBLIS2S26\linguagem-blis2s26 | fast: False


## 1. N-gramas de caracteres e *hashing*

Exemplo do artigo: `where` com $n=3$ → `<wh, whe, her, ere, re>` + `<where>`. Note que `her` (dentro de *where*) é diferente de `<her>` (a palavra *her*) graças aos delimitadores.

In [2]:
from replang.models.fasttext_np import char_ngrams, fnv1a
for w in ["where", "onde", "casinha", "incrivelmente"]:
    grams = char_ngrams(w, 3, 6)
    print(f"{w:<14} {len(grams):>2} n-gramas: {grams[:8]}{' …' if len(grams) > 8 else ''}")
print("\nhash FNV-1a (mod 2·10⁶):", {g: fnv1a(g) % 2_000_000 for g in char_ngrams("casa", 3, 4)})

where          14 n-gramas: ['<wh', 'whe', 'her', 'ere', 're>', '<whe', 'wher', 'here'] …
onde           10 n-gramas: ['<on', 'ond', 'nde', 'de>', '<ond', 'onde', 'nde>', '<onde'] …
casinha        22 n-gramas: ['<ca', 'cas', 'asi', 'sin', 'inh', 'nha', 'ha>', '<cas'] …
incrivelmente  46 n-gramas: ['<in', 'inc', 'ncr', 'cri', 'riv', 'ive', 'vel', 'elm'] …

hash FNV-1a (mod 2·10⁶): {'<ca': 916747, 'cas': 1956174, 'asa': 396164, 'sa>': 331429, '<cas': 545640, 'casa': 18301, 'asa>': 905934}


## 2. Implementação *from scratch*

`replang.models.fasttext_np.FastText` herda do Skip-gram numpy e troca o vetor de entrada pela **média** dos vetores dos n-gramas (+ palavra). O gradiente de cada par é distribuído por todos os n-gramas da palavra central — é assim que *casinha* "empresta" o que aprendeu a *casa*.

In [3]:
import inspect, time
from replang.models.fasttext_np import FastText
src = inspect.getsource(FastText.train)
print(src[src.index("centers = np.array"):src.index("losses.append")])

centers = np.array([p[0] for p in batch])
                targets = np.array([p[1] for p in batch])
                ng, m, ln = self.ng_arr[centers], self.ng_mask[centers], self.ng_len[centers]
                h = (self.W_in[ng] * m[:, :, None]).sum(1) / ln[:, None]
                loss, grad_h = self._step_neg(h, targets, lr)
                g = (grad_h / ln[:, None])[:, None, :] * m[:, :, None]
                np.add.at(self.W_in, ng.ravel(), -g.reshape(-1, self.dim))
                


In [4]:
from replang.data.corpora import load_machado_sentences
sents = load_machado_sentences()
sub = sents[:12000 if FAST else 25000]
t = time.time()
ft = FastText(dim=50, window=3, epochs=2, negative=5, min_count=3, bucket=100_000, batch_size=256, seed=1).train(sub)
wv_ft = ft.to_wordvectors("fastText numpy (40k sentenças)")
print(f"{time.time()-t:.0f}s | |V| = {len(wv_ft):,} | perda final {ft.history[-1]['loss']:.3f}")
for w in ["amor", "triste", "casa", "capitu"]:
    print(f"{w:<8} {[x for x, _ in wv_ft.most_similar(w, topn=6)] if w in wv_ft else '(fora do vocabulário do subconjunto — mas o fastText ainda dá um vetor; veja abaixo)'}")

89s | |V| = 10,423 | perda final 2.814
amor     ['confirmou', 'desfazer', 'sentimento', 'sente', 'sentindo', 'sentou']
triste   ['entristecia', 'prudência', 'tristemente', 'resistência', 'persistente', 'naturalmente']
casa     ['casal', 'casar', 'casava', 'casaria', 'casais', 'casara']
capitu   (fora do vocabulário do subconjunto — mas o fastText ainda dá um vetor; veja abaixo)


### 2.1 Vetores para palavras **fora do vocabulário**

In [5]:
for oov in ["tristíssimo", "capituzinha", "amorzão", "desamoroso", "xyzzyq"]:
    v = ft.word_vector(oov)
    sims = wv_ft.similarities(v)
    print(f"{oov:<14} OOV → vizinhos: {[wv_ft.words[i] for i in np.argsort(-sims)[:6]]}")

tristíssimo    OOV → vizinhos: ['principalmente', 'encontravam', 'apresentaram', 'expediente', 'maquinalmente', 'vieram']
capituzinha    OOV → vizinhos: ['rematar', 'minha', 'capital', 'pratinha', 'concurso', 'prestou']
amorzão        OOV → vizinhos: ['sentimento', 'sentou', 'ressentimento', 'parecem', 'conhecia', 'consolar']
desamoroso     OOV → vizinhos: ['esperançado', 'correio', 'desespero', 'amoroso', 'abraçando', 'desesperado']
xyzzyq         OOV → vizinhos: ['nababo', 'se', 'judeu', 'que', 'gozo', 'nº']


### 2.2 Quais n-gramas importam? (§6.2 e Tabela 6 do artigo)

O artigo remove um n-grama de cada vez e mede o quanto a representação muda. Os n-gramas mais importantes tendem a ser **morfemas**: para *Autofahrer*, `Auto` e `fahrer`; para *kindness*, `ness>`.

In [6]:
from replang.viz import ngram_bar
for w in ["tristeza", "casinha", "amorosamente"]:
    imp = ft.ngram_importance(w)
    print(f"{w:<13} n-gramas mais importantes: {[g for g, _ in imp[:6]]}")
ngram_bar(ft.ngram_importance("amorosamente")[:15], title="'amorosamente': queda do cosseno ao remover cada n-grama (maior = mais importante)").show()

tristeza      n-gramas mais importantes: ['<tr', 'ist', 'ris', 'eza', '<tri', 'iste']
casinha       n-gramas mais importantes: ['<ca', '<cas', 'nha', 'inh', 'ha>', 'inha']
amorosamente  n-gramas mais importantes: ['te>', 'ros', 'nte', 'rosa', 'moros', 'orosam']


## 3. Modelo completo (gensim) no corpus Machado: fastText × Skip-gram

`machado_ft100` (gensim, Skip-gram, 100d, n ∈ [3,6]) e `machado_sg100` foram treinados com os mesmos hiperparâmetros. Comparamos:
1. vizinhos de palavras raras e OOV;
2. pares morfológicos (similaridade esperada alta);
3. analogias **sintáticas** × **semânticas** (Tabela 2 do artigo: subpalavras ajudam na sintaxe e podem atrapalhar na semântica).

In [7]:
from replang.data.embeddings import load_local_model
from replang.models.trainers import load_fasttext_model
sg = load_local_model("machado_sg100")
ftm = load_fasttext_model("machado_ft100")
wv_ftm = load_local_model("machado_ft100")
raras = ["capituzinha", "escravocrata", "tristíssimo", "desconfiadíssimo", "aborrecimento", "cavalheirismo"]
for w in raras:
    nn_ft = [x for x, _ in ftm.wv.most_similar(w, topn=5)]
    nn_sg = [x for x, _ in sg.most_similar(w, topn=5)] if w in sg else "OOV (sem vetor)"
    print(f"{w:<18} fastText: {nn_ft}\n{'':<18} Skip-gram: {nn_sg}")

capituzinha        fastText: ['angrazinha', 'sinhazinha', 'coitadinha', 'rochinha', 'luizinha']
                   Skip-gram: OOV (sem vetor)
escravocrata       fastText: ['escravo', 'escravidão', 'homeopata', 'leonarda', 'leopoldina']
                   Skip-gram: OOV (sem vetor)
tristíssimo        fastText: ['riquíssimo', 'tristíssima', 'triptolemo', 'veríssimo', 'vigésimo']
                   Skip-gram: ['planeado', 'correspondendo', 'piauí', 'tirou-me', 'presidencial']
desconfiadíssimo   fastText: ['desconfio', 'desconfia', 'desconfiara', 'desconfiei', 'desconfiado']
                   Skip-gram: OOV (sem vetor)
aborrecimento      fastText: ['esclarecimento', 'desvanecimento', 'atrevimento', 'esquecimento', 'acolhimento']
                   Skip-gram: ['importava', 'desengano', 'ressentimento', 'vexame', 'repulsa']
cavalheirismo      fastText: ['supérfluo', 'catolicismo', 'eufemismo', 'equívoco', 'genuíno']
                   Skip-gram: OOV (sem vetor)


In [8]:
pares = [("menino", "menininho"), ("casa", "casinha"), ("amor", "amoroso"), ("escrever", "escrevia"), ("feliz", "felicidade"), ("triste", "tristeza"), ("rei", "rainha"), ("gato", "gatinho")]
rows = [{"par": f"{a} – {b}", "fastText": float(ftm.wv.similarity(a, b)), "Skip-gram": sg.similarity(a, b) if sg.has(a, b) else np.nan} for a, b in pares]
pd.DataFrame(rows).style.format({"fastText": "{:.2f}", "Skip-gram": "{:.2f}"})

,par,fastText,Skip-gram
0,menino – menininho,0.91,nan
1,casa – casinha,0.84,0.75
2,amor – amoroso,0.78,0.67
3,escrever – escrevia,0.85,0.70
4,feliz – felicidade,0.66,0.70
5,triste – tristeza,0.81,0.70
6,rei – rainha,0.40,0.79
7,gato – gatinho,0.81,nan


In [9]:
from replang.data.analogies import load_analogies
from replang.eval.analogies import evaluate_analogies
an = load_analogies("pt-br")
rows = []
for wv in [sg, wv_ftm]:
    r = evaluate_analogies(wv, an, restrict=None).set_index("categoria")
    rows.append({"modelo": wv.name, "sintática": r.loc["TOTAL sintática", "acurácia"], "semântica": r.loc["TOTAL semântica", "acurácia"], "cobertas": int(r.loc["TOTAL", "cobertas"])})
pd.DataFrame(rows).style.format({"sintática": "{:.1%}", "semântica": "{:.1%}"})

,modelo,sintática,semântica,cobertas
0,machado_sg100,2.8%,16.5%,3771
1,machado_ft100,45.3%,21.1%,3771


## 4. Efeito do tamanho do corpus (§5.4 e Fig. 1 do artigo)

O artigo mostra que o fastText **satura cedo**: com 5 % da Wikipédia alemã supera o CBOW com 100 %. Reproduzimos a tendência treinando os dois modelos (gensim, 50d, 3 épocas) em 5 %, 20 % e 100 % do corpus Machado e medindo as analogias **sintáticas/morfológicas** do LX-4WAnalogies (plural, particípio, passado, plural de verbos) — a competência que os n-gramas capturam.

In [10]:
from gensim.models import Word2Vec as GW2V, FastText as GFT
cats = ["gram5-present-participle", "gram7-past-tense", "gram8-plural", "gram9-plural-verbs"]
fracs = [0.05, 0.2, 1.0] if not FAST else [0.05, 0.2]
rows = []
for frac in fracs:
    sub = sents[: int(len(sents) * frac)]
    for name, cls, kw in [("Skip-gram", GW2V, {}), ("fastText", GFT, {"min_n": 3, "max_n": 6, "bucket": 100_000})]:
        t = time.time()
        m = cls(sub, vector_size=50, window=5, sg=1, negative=5, min_count=5, epochs=3, sample=1e-4, workers=settings.workers, seed=1, **kw)
        wv = WordVectors.from_gensim(m.wv, name)
        r = evaluate_analogies(wv, an, restrict=None).set_index("categoria")
        cob = int(r.loc[cats, "cobertas"].sum()); ac = int(r.loc[cats, "acertos"].sum())
        rows.append({"fração": frac, "tokens": sum(map(len, sub)), "modelo": name, "|V|": len(wv), "questões cobertas": cob, "acurácia sintática": ac / max(cob, 1), "s": round(time.time() - t)})
df = pd.DataFrame(rows)
df.style.format({"acurácia sintática": "{:.1%}"})

,fração,tokens,modelo,|V|,questões cobertas,acurácia sintática,s
0,0.050000,118626,Skip-gram,2517,152,0.0%,0
1,0.050000,118626,fastText,2517,152,7.9%,0
2,0.200000,480259,Skip-gram,7437,706,0.0%,1
3,0.200000,480259,fastText,7437,706,2.4%,1
4,1.000000,2357192,Skip-gram,23194,2616,1.1%,3
5,1.000000,2357192,fastText,23194,2616,21.4%,6


In [11]:
import plotly.express as px
px.line(df, x="tokens", y="acurácia sintática", color="modelo", markers=True, log_x=True, template="plotly_white",
        title="Analogias morfológicas × tamanho do corpus: fastText aprende com muito menos dados (cf. Fig. 1 de Bojanowski et al.)").show()

## 5. O que o artigo encontrou

* **Similaridade de palavras** (Tabela 1): `sisg` vence `sg`/`cbow` em 8 de 9 línguas; ganho maior em alemão, árabe, russo; empate em inglês (WS353), onde as palavras são comuns.
* **Analogias** (Tabela 2): grande ganho nas **sintáticas** (tcheco 52,8 → 77,8; alemão 44,5 → 56,4); nas semânticas, igual ou pior.
* **Tamanho dos n-gramas** (Tabela 4): 3–6 é razoável; 2-gramas não ajudam (um caractere + um delimitador).
* **Modelagem de linguagem** (Tabela 5): inicializar um LSTM com `sisg` reduz a perplexidade (−8 % tcheco, −13 % russo).
* **Hartmann et al. (2017)**: em PT, fastText foi o melhor nas analogias sintáticas (58,7 % com Skip-gram 300d) e surpreendentemente fraco em POS tagging (a tokenização dos clíticos é apontada como causa).

### Pontos para a apresentação
1. A hipótese distribucional continua valendo — mas agora **compartilhada entre formas**: a distribuição de `<cas` informa *casa*, *casas*, *casinha*.
2. OOV deixa de ser um buraco: vetores para erros de digitação, neologismos e nomes raros.
3. O custo: n-gramas espúrios (`ame` em *amesma*?) e perda em analogias semânticas.
4. É a ponte para o **ELMo** (notebook 12), que usa uma CNN de caracteres como camada de entrada.

### Perguntas para discussão
* Por que o *hashing* funciona apesar das colisões? (2 M buckets × n-gramas raros; colisões são ruído distribuído)
* O que o fastText faz com *manga* (fruta) e *manga* (camisa)? (nada — ainda é um vetor por forma; gancho para ELMo)

## Referências
{refs("fasttext", "mikolov13b", "hartmann", "elmo")}